# 5-HT2A Membrane Build — Amber route (Option A, works with packmol natively)

After the packmol↔CHARMM36 template incompatibility, this uses packmol-memgen's **native Amber parameterisation** (`--parametrize` with ff19SB + Lipid21). tleap handles the lipid/water/ion connectivity directly — no template-matching battle. Produces Amber `.prmtop`/`.inpcrd`, which OpenMM reads via ParmEd.

**Force-field note for the paper:** D2 used CHARMM36; 5-HT2A here uses **Amber ff19SB / Lipid21 / TIP3P**. State this in Methods as a builder-driven choice. Both are validated membrane force fields; the comparison remains valid as each system is internally consistent.

**Key fix vs the failed CHARMM attempts:** we let packmol **protonate and cap termini** (drop `--notprotonate`). The ICL3 chain break caused an OXT/terminus error in both CHARMM and (per the Amber mailing list) tleap; letting packmol cap the ends avoids it.

**Run in the `packing` env** (has AmberTools/tleap). Equilibration cells switch to `mddock` for OpenMM, OR run OpenMM from `packing` if it has openmm — check.

---
## Part 1 — Re-pack AND parametrise in one step (Amber)

In [ ]:
# Cell 1 (BASH, run in terminal with: conda activate packing):
#
#   cd /workspace/5ht2a_system
#   packmol-memgen --pdb 5ht2a_oriented_clean.pdb --lipids DPPC \
#       --salt --salt_c Na+ --saltcon 0.15 --dist 14 --dist_wat 17.5 \
#       --preoriented --parametrize --ffprot ff19SB --fflip lipid21 --ffwat tip3p \
#       --overwrite --log packmol_amber.log
#
# Notes:
#  - NO --notprotonate this time: packmol protonates + caps termini (avoids the
#    OXT/terminus tleap error at the ICL3 break).
#  - NO --nottrim: let it tidy the structure for tleap.
#  - --preoriented: reuse the OPM orientation already in 5ht2a_oriented_clean.pdb.
#  - Outputs: bilayer_5ht2a_oriented_clean.prmtop + .inpcrd  (Amber topology+coords)
#
# This cell is a REMINDER. Run the command above in the terminal, then continue below.
print("Run the packmol-memgen --parametrize command in a terminal (see comment above).")
print("Expected outputs: a .prmtop and .inpcrd (or .crd) file in /workspace/5ht2a_system/")


In [ ]:
# Cell 2: locate the Amber topology/coord files tleap produced
import glob, os
os.chdir("/workspace/5ht2a_system")
prm = sorted(glob.glob("*.prmtop")) + sorted(glob.glob("*.top"))
crd = sorted(glob.glob("*.inpcrd")) + sorted(glob.glob("*.crd") ) + sorted(glob.glob("*.rst7"))
print("prmtop/top candidates:", prm)
print("inpcrd/crd candidates:", crd)
print("\nPick the SYSTEM files (protein+lipid+water+ions), not lipid-only ones.")
print("Typically named like bilayer_5ht2a_oriented_clean.prmtop / .inpcrd")


**Send Claude the file list from Cell 2** if the names are ambiguous — we want the full-system `.prmtop`/`.inpcrd`, not any lipid-only intermediate (`*_lipid.top`). tleap sometimes writes both.

---
## Part 2 — Load into OpenMM via ParmEd and equilibrate (run in mddock)

In [ ]:
# Cell 3: load Amber prmtop/inpcrd into OpenMM (switch kernel to mddock if needed)
import os, sys, time, pickle
from pathlib import Path
import numpy as np
import parmed as pmd
from openmm import app, unit as u
from openmm import (LangevinMiddleIntegrator, MonteCarloMembraneBarostat, Platform)
os.chdir("/workspace/5ht2a_system")

PRMTOP = "bilayer_5ht2a_oriented_clean_lipid.top"
INPCRD = "bilayer_5ht2a_oriented_clean_lipid.crd"
assert Path(PRMTOP).exists() and Path(INPCRD).exists(), "Set PRMTOP/INPCRD to the Cell 2 names"

amber = pmd.load_file(PRMTOP, INPCRD)
print(f"Loaded {len(amber.atoms):,} atoms")
from collections import Counter
print("Residues:", dict(Counter(r.name for r in amber.residues).most_common(8)))
print("Box:", amber.box)


In [ ]:
# Cell 4: build OpenMM system (Amber FF already baked into prmtop) + membrane barostat
system = amber.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1.0*u.nanometer,
    constraints=app.HBonds, rigidWater=True, removeCMMotion=True)
system.addForce(MonteCarloMembraneBarostat(1.0*u.bar, 0.0*u.bar*u.nanometer, 310*u.kelvin,
    MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25))
integ = LangevinMiddleIntegrator(310*u.kelvin, 1.0/u.picosecond, 0.002*u.picosecond)
sim = app.Simulation(amber.topology, system, integ, Platform.getPlatformByName("CUDA"),
                     {"Precision":"mixed","DeviceIndex":"0"})
sim.context.setPositions(amber.positions)
print(f"✓ System: {system.getNumParticles():,} particles on CUDA")


In [ ]:
# Cell 5: minimise
e0=sim.context.getState(getEnergy=True).getPotentialEnergy()
print("PE before:", e0)
sim.minimizeEnergy(maxIterations=10000, tolerance=10*u.kilojoule_per_mole/u.nanometer)
e1=sim.context.getState(getEnergy=True).getPotentialEnergy()
print("PE after :", e1)
with open("ht2a_minimized.pdb","w") as f:
    app.PDBFile.writeFile(amber.topology, sim.context.getState(getPositions=True).getPositions(), f)
print("✓ minimised")


In [ ]:
# Cell 6: restrained NPT (1 ns) -> free NPT (2 ns)
from openmm import CustomExternalForce
solvent={"WAT","HOH","Na+","Cl-","K+","NA","CL"}
r=CustomExternalForce("0.5*k*periodicdistance(x,y,z,x0,y0,z0)^2")
r.addGlobalParameter("k",1000.0*u.kilojoule_per_mole/u.nanometer**2)
for p in ("x0","y0","z0"): r.addPerParticleParameter(p)
pos=sim.context.getState(getPositions=True).getPositions(asNumpy=True)
heavy=[a.index for a in amber.topology.atoms()
       if a.element is not None and a.element.symbol!="H" and a.residue.name not in solvent]
for idx in heavy:
    p=pos[idx].value_in_unit(u.nanometer); r.addParticle(idx,[p[0],p[1],p[2]])
ri=system.addForce(r); sim.context.reinitialize(preserveState=True)
sim.context.setVelocitiesToTemperature(310*u.kelvin)
sim.reporters.append(app.StateDataReporter(sys.stdout,25000,step=True,time=True,
    temperature=True,volume=True,speed=True,separator=" | "))
print(f"Restrained 1 ns ({len(heavy)} atoms)...")
sim.step(500_000); sim.saveState("ht2a_equil_restrained.xml")
system.removeForce(ri); sim.context.reinitialize(preserveState=True)
sim.reporters.clear()
sim.reporters.append(app.StateDataReporter(sys.stdout,25000,step=True,time=True,
    temperature=True,density=True,speed=True,separator=" | "))
print("Free 2 ns...")
sim.step(1_000_000)
sim.saveState("ht2a_final_equilibrated.xml")
print("✓ ht2a_final_equilibrated.xml")


In [ ]:
# Cell 7: save ParmEd structure + docking receptor for Notebook 2
st=sim.context.getState(getPositions=True)
amber.coordinates = st.getPositions(asNumpy=True).value_in_unit(u.angstrom)
with open("ht2a_gmx_top.pkl","wb") as f: pickle.dump(amber,f)
print(f"✓ ht2a_gmx_top.pkl ({len(amber.atoms):,} atoms)")
standard=set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())
prot_idx=[a.idx for a in amber.atoms if a.residue.name in standard or a.residue.name in
          ("HID","HIE","HIP","CYX","ASH","GLH","LYN")]   # Amber protonation-variant names
amber[prot_idx].save("ht2a_equilibrated_protein.pdb", overwrite=True)
print("✓ ht2a_equilibrated_protein.pdb")
print("\nStage 2 (Amber) COMPLETE. Notebook 2 inputs ready.")


In [ ]:
import pickle
with open("/workspace/5ht2a_system/ht2a_gmx_top.pkl","rb") as f:
    s = pickle.load(f)

asps = [(r.number, r.name) for r in s.residues if r.name in ("ASP","ASH")]
print(f"Aspartates: {len(asps)}")
print("First few:", asps[:10])
near155 = [(r.number, r.name) for r in s.residues if r.name in ("ASP","ASH") and 150 <= r.number <= 160]
print("ASP near 155:", near155)

# Also show the protein residue numbering range, since tleap may have renumbered
prot = [r.number for r in s.residues if r.name in 
        set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL CALA CTYR".split())]
print(f"Protein residue number range: {min(prot)} to {max(prot)}")

In [ ]:
import pickle
with open("/workspace/5ht2a_system/ht2a_gmx_top.pkl","rb") as f:
    s = pickle.load(f)

standard = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL CALA CTYR".split())
prot = [r for r in s.residues if r.name in standard]

# Original receptor segments: 69-265 then 313-399 (ICL3 gap removed)
orig_numbers = list(range(69, 266)) + list(range(313, 400))
print(f"Protein residues in structure: {len(prot)}, expected original count: {len(orig_numbers)}")

# Build mapping: new index -> original number
mapping = {}
for new_res, orig_num in zip(prot, orig_numbers):
    mapping[orig_num] = (new_res.number, new_res.name)

# Check the key residues
for orig in [155, 336, 332, 242, 159]:
    if orig in mapping:
        newnum, name = mapping[orig]
        print(f"  Original {orig} -> new residue {newnum} ({name})")

---
## Notes

- **Amber protonation residue names:** ff19SB renames some residues (HIS->HID/HIE/HIP, CYS->CYX for disulfides, etc.). Cell 7 includes these in the protein selection. Notebook 2's docking receptor extraction may need the same name set — flag Claude if the receptor PDB looks short.
- **Notebook 2 compatibility:** Notebook 2 merges the racetam ligand (OpenFF) with this structure. Since the membrane is now Amber (not CHARMM36), the merge still works at the ParmEd level — ParmEd handles mixed parameter sources — but the ligand uses OpenFF as before, consistent with the D2 runs. The receptor+lipid being Amber vs CHARMM36 is the only cross-receptor FF difference, noted in Methods.
- **If tleap errored** during Cell 1's packmol --parametrize (e.g. the OXT terminus error at the ICL3 break), send Claude the tleap log — the fix is usually capping the break ends, which packmol's protonation should handle now that we dropped --notprotonate.